In [ ]:
from contextlib import asynccontextmanager
from typing import List
from fastapi import FastAPI, HTTPException, Request, BackgroundTasks
from schemas import TransactionSchema, PredictionResponseSchema
from starlette.concurrency import run_in_threadpool
import joblib
import pandas as pd
from uuid import uuid4
import datetime

def valid_transaction():
    return {
        "user_id": 42,
        "amount": 1500.50,
        "merchant_category": "grocery",
        "location_lat": 50.45,
        "location_lon": 30.52,
        "is_international": False,
    }

In [3]:
valid_transaction1 = valid_transaction()

In [10]:
valid_transaction1

{'user_id': 42,
 'amount': 1500.5,
 'merchant_category': 'grocery',
 'location_lat': 50.45,
 'location_lon': 30.52,
 'is_international': False}

In [ ]:
model = joblib.load("../models/model.joblib")
    
proba = model.predict_proba(pd.DataFrame([valid_transaction1]))
fraud_probability = proba[0][1]
is_fraud = fraud_probability >= 0.5
risk_level = "high" if fraud_probability >= 0.7 else "medium" if fraud_probability >= 0.3 else "low"

res = PredictionResponseSchema(
    transaction_id=uuid4(),
    timestamp=datetime.datetime.now(tz=datetime.timezone.utc),
    is_fraud=is_fraud,
    fraud_probability=fraud_probability,
    risk_level=risk_level
)
res